# Cuaderno B: limpieza, split, baselines, BERT y evaluación

**Requisitos:** haber terminado el cuaderno A (descarga + OCR). Los datos están en tu Drive, en `cv_experience_ocr_bert_data`.

⚠️ **NO instales PaddlePaddle aquí.** Este cuaderno usa PyTorch, que choca con Paddle en la misma sesión.

Antes de empezar:
1. `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar` (la GPU solo hace falta para BERT y los embeddings; las demás celdas funcionan igual).
2. En 🔑 **Secretos**: `HF_TOKEN` (con acceso al notebook activado) y `GH_TOKEN` si tu repo es privado.
3. Ejecuta las celdas **en orden**.

## 1. GPU, Drive y repositorio

In [ ]:
!nvidia-smi | head -n 12

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from google.colab import userdata

GITHUB_USER = "patriciacastr"
REPO = "HW_04_202602_cv_experience_ocr_bert"
BRANCH = "main"

try:
    gh = userdata.get("GH_TOKEN")
    url = f"https://{gh}@github.com/{GITHUB_USER}/{REPO}.git"
except Exception:
    url = f"https://github.com/{GITHUB_USER}/{REPO}.git"

if not os.path.exists(f"/content/{REPO}"):
    !git clone -b {BRANCH} {url} /content/{REPO}
else:
    !cd /content/{REPO} && git pull origin {BRANCH}
%cd /content/{REPO}

## 2. Instalar librerías (PyTorch ya viene con Colab)

In [ ]:
!pip install -q transformers sentence-transformers scikit-learn pandas matplotlib jiwer joblib
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
import torch
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 3. Enlazar carpetas con Drive
`data/` (imágenes y OCR del cuaderno A), `models/` y `output/` viven en Drive, así nada se pierde si Colab se desconecta.

In [ ]:
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))   # para descargar los modelos base

DRIVE = "/content/drive/MyDrive/cv_experience_ocr_bert_data"
for name in ["raw", "ocr", "processed", "splits"]:
    os.makedirs(f"{DRIVE}/{name}", exist_ok=True)
    !rm -rf /content/{REPO}/data/{name}
    !ln -s {DRIVE}/{name} /content/{REPO}/data/{name}
for name in ["models", "output"]:
    os.makedirs(f"{DRIVE}/{name}", exist_ok=True)
    !rm -rf /content/{REPO}/{name}
    !ln -s {DRIVE}/{name} /content/{REPO}/{name}
!ls data/raw | head

## 4. Calidad del OCR (CER, WER, años leídos bien)
Guarda tablas en `output/tables` y la figura en `output/figures`.

In [ ]:
!python src/ocr_quality.py

## 5. Revisar 50 textos OCR a mano (lo exige la tarea)
Lee los textos y busca frases que den la respuesta (años de experiencia, edad, fecha de nacimiento).
Si encuentras alguna que el enmascarado no cubra, **avísame** y ajustamos los patrones de `preprocessing.py`.

In [ ]:
!python src/preprocessing.py --review 50

## 6. Limpieza y enmascarado, y split 70/15/15

In [ ]:
!python src/preprocessing.py --conf-threshold 0.6
!python src/split.py

## 7. Baselines (se entrenan en segundos o pocos minutos)

In [ ]:
!python src/baseline_median.py
!python src/baseline_tfidf.py
!python src/baseline_tfidf.py --keep-digits 0

In [ ]:
!python src/baseline_embeddings.py

## 8. BERT
**Primero la prueba rápida** (1 época, 64 CVs): confirma que todo corre antes de gastar GPU.

In [ ]:
!python src/train_bert.py --limit 64 --epochs 1

**Modelo principal** (estrategia `headtail`, pérdida MSE). Puede tardar; revisa el MAE de validación por época.

In [ ]:
!python src/train_bert.py

## 9. Variantes para la Pregunta 3 (todas, una tras otra, en segundo plano)
Con `max-len 512` ningún CV se corta, así que las estrategias de texto largo se comparan forzando `max-len 128`. Tarda unos 10-12 min en total.

In [ ]:
!nohup sh -c "python -u src/train_bert.py --loss huber --name bert_huber; python -u src/train_bert.py --strategy head --max-len 128 --name bert_head128; python -u src/train_bert.py --strategy headtail --max-len 128 --name bert_headtail128; python -u src/train_bert.py --strategy chunks --max-len 128 --name bert_chunks128; python -u src/train_bert.py --text-col text_raw --name bert_raw" > bert_variants.log 2>&1 &

In [ ]:
# Ejecuta cada pocos minutos hasta ver las 5 líneas [bert_...] y "TERMINÓ"
!grep -E "^\[bert" bert_variants.log
!pgrep -af train_bert.py | grep -v pgrep || echo "TERMINÓ / NO CORRE"

## 9b. Referencia por reglas y prueba de fechas (Pregunta 1)
La referencia suma las duraciones de los cargos (sin aprender). La prueba mueve todas las fechas +5 años y borra un cargo en 20 CVs del test. **Ejecutar después de que BERT y los baselines estén entrenados.**

In [ ]:
!python src/baseline_rules.py

In [ ]:
!python src/probe_dates.py --n 20

## 10. Evaluación final (tablas y figuras en `output/`)

In [ ]:
!python src/evaluation.py

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
display(pd.read_csv("output/tables/model_comparison.csv").round(3))

In [ ]:
from IPython.display import Image, display
display(Image("output/figures/pred_vs_true.png"))
display(Image("output/figures/mae_by_band.png"))

## 11. Llevar los resultados al repositorio
Las tablas, métricas y figuras están en tu Drive (`cv_experience_ocr_bert_data/output`).
Descárgalas y súbelas al repo desde el Codespace, en una rama con su Pull Request.
No subas `models/` (son pesos) ni `data/`.